# 🧔‍♂️🎮👩 Gender Representation in Video Games 👩🎮🧔‍♂️ 

## Context:
These are top-selling or top-rated games, so the numbers describe successful games, not the whole industry.

## Objective: Analizing the representation of women in Videogames. This analyze will be run through the lense of 3 main questions :

| **Q1** | Are women more sexualized than men, in what ways, and does it depend on the role they play ? 
| **Q2** | What roles do women hold: lead, playable, romantic interest? 
| **Q3** | Good vs. bad: are women allies, antagonists or main villains? 

### Angles that can be added to each question

Each of Q1–Q3 can be re-read through these three lenses:

| **A. Time** | Has it changed over time (2012–2022)? 
| **B. Team mix** | Does the team's gender mix change what is shown?
| **C. Country** | Does the developer's country change what is shown?

# 📥 Data Loading

### 🎯 Goal
Understand what each table contains (size, columns, types, typical values) before cleaning anything.

### 📥 Input
Three CSV files downloaded with `kagglehub` from the Kaggle dataset `br33sa/gender-representation-in-video-games`:

| Table | One row is... | Links to |
|---|---|---|
| `games` | a game | `characters.Game` |
| `characters` | a relevant character | `games.Game_Id`, `sexualization.Id` |
| `sexualization` | the sexualization rating of a character | `characters.Id` |

### 🛠️ What we do
1. Import the libraries (`pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`, `kagglehub`, `os`) and set the plot defaults: `sns.set_style("whitegrid")` and figure size `(10, 6)`.
2. Download the dataset with `kagglehub.dataset_download("br33sa/gender-representation-in-video-games")` and store the folder in `path`.
3. Read each CSV with `pd.read_csv(os.path.join(path, ...))` into `games`, `characters` and `sexualization`.
4. For each table, check `shape`, `columns.tolist()`, `head()` and `info()`.
5. Summarize with `describe()` (numeric columns) and `describe(exclude='number')` (text columns).

### 📝 Observations
*(Fill in after running the code.)*
- e.g. `Unnamed: 27` is completely empty: a column to drop.
- e.g. `Percentage_non_male` and `Team_percentage` are stored as text.
- Full column definitions: `docs/data-dictionary.md`.

## 1. Imports and plot defaults

In [ ]:
#installing compatible version of plotly for colab, and getting the repo (the cleaning functions are in src/cleaning.py)
import sys
if "google.colab" in sys.modules:
    %pip install -q plotly==7.1.0
    import os
    if not os.path.exists("EDA_Manon"):
        !git clone -q https://github.com/IA-P1-BCN/EDA_Manon
    %cd EDA_Manon

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from IPython.display import display, Markdown
import plotly.express as px

# Cleaning functions live in src/cleaning.py (board #24)
from src.cleaning import (
    drop_empty_games_columns, drop_unused_games_columns, drop_age, drop_review_sites, fill_series,
    rename_columns, fix_release_date, fix_percentages, fix_son_damsel_flag,
    add_is_sexualized, add_role_tier, add_has_female_team, add_team_group,
    add_share_women, add_share_sexualized, add_region, read_raw_tables, clean_all)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


## 2. Download the dataset

In [ ]:
path = kagglehub.dataset_download("br33sa/gender-representation-in-video-games")
print("path to files:", path)

## 3. Read each CSV

In [ ]:
games = pd.read_csv(os.path.join(path, "games.grivg.csv"))
characters = pd.read_csv(os.path.join(path, "characters.grivg.csv"))
sexualization = pd.read_csv(os.path.join(path, "sexualization.grivg.csv"))

tables = {"games": games, "characters": characters, "sexualization": sexualization}

descriptions = {
    "games": "Information about the games. Each row represents a game. Includes genre, development team, and critic reviews.",
    "characters": "Information about the characters. Each row represents a relevant character. Joins with `games` by `Game`.",
    "sexualization": "Information about the sexualization ratings of characters. Each row represents a rating for a character. Joins by `Id`.",
}

## 4. Check structure of each table

In [ ]:
for name, d in tables.items():
    display(Markdown(f"#### {name}\n\n{descriptions[name]}"))
    print(f"shape: {d.shape}\n")
    print(f"columns: {d.columns.tolist()}\n")
    display(d.head())
    d.info()
    print("\n")

## 5. Descriptive statistics

In [ ]:
for name, d in tables.items():
    display(Markdown(f"#### {name}"))
    display(d.describe())                       # numeric columns
    display(d.describe(exclude="number"))       # text columns

# 🔍 Exploración

> We look for problems in the data (missing values, duplicates, outliers, inconsistencies).

---

### 🎯 Goal
Find what needs cleaning before changing anything. Studying the variables comes later, in *Análisis*, once the tables are clean.

### 📥 Input
The three tables loaded in *Carga* (`games`, `characters`, `sexualization`), still unmodified.

### 🛠️ What we do
1. **Missing values:** count nulls per column (`df.isnull().sum()`) and show the percentage (`nulls / len(df) * 100`), only for columns above 0. Optional: `missingno` matrix.
2. **Duplicates:** `df.duplicated().sum()`.
3. **Outliers:** IQR rule, flagged outside `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`, plus boxplots.
4. **Dubious data:** check consistency between columns (e.g. `characters.Sexualization` vs `sexualization.Total`) and impossible values (e.g. ages).

### 📊 Charts checklist
Every chart has a title, named axes, and a legend when it has more than one series. Minimum **5 visualizations** in total.

### 📝 Observations and decisions
*(Fill in after running the code. Nothing is changed in this section: decisions are applied in *Limpieza*.)*

| Finding | Where | What we will do | Why |
|---|---|---|---|
| *(e.g. `Series` has 58% nulls)* | `games` | *(keep / fill / drop)* | *(reason)* |
| *(e.g. outlier in `Total_team`)* | `games` | *(keep / cap / drop)* | *(reason)* |
| 5 games above the IQR limit in `Team_percentage` (43% to 67%) | `games` | keep | real games with balanced teams, not errors |
| 1 to 5 games flagged in `Relevant_males`, `Relevant_no_males`, `Percentage_non_male`, `Female_team` | `games` | keep | real values: large casts, all-female casts, small teams with several women |
| `Protagonist` and `Sexualization` flagged massively by the IQR rule | `games`, `characters` | keep | the IQR is 0, so every value other than the usual one is flagged; they are valid counts and scores |

## 1. Missing values

In [ ]:
import missingno as msno

for name, d in tables.items():
    display(Markdown(f"#### {name}"))

    nulls = d.isnull().sum()
    nulls = nulls[nulls > 0]

    if nulls.empty:
        print("No missing values.\n")
        continue

    report = pd.DataFrame({
        "missing": nulls,
        "percent": (nulls / len(d) * 100).round(1),
    }).sort_values("percent", ascending=False)
    display(report)

    ax = msno.matrix(d, sparkline=False)
    ax.set_title(f"Missing values in {name}")
    plt.show()



## 2. Duplicates

In [ ]:
for name, d in tables.items():
    display(Markdown(f"#### {name}"))
    n_duplicates = d.duplicated().sum()

    if n_duplicates == 0:
        print("No duplicates found.\n")
        continue

    print(f"{n_duplicates} duplicated row(s):")
    display(d[d.duplicated(keep=False)]) 
    tables[name] = d.drop_duplicates()

## 3. Outliers

**Method:** IQR rule. A value is an outlier if it lies outside `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`, checked with a table and boxplots.

### Detection on the numeric variables of `games`

`Team_percentage` and `Percentage_non_male` are still text (`"11%"`) at this stage, so we convert them on a local copy only. The raw tables stay untouched.


In [ ]:
games = tables["games"].copy()

# Still text ("11%") at this stage: convert on the local copy only
for col in ["Percentage_non_male", "Team_percentage"]:
    games[col] = games[col].astype(str).str.rstrip("%").astype(float)

numeric_cols = ["Protagonist", "Relevant_males", "Relevant_no_males", "Percentage_non_male",
                "Total_team", "female_team", "Team_percentage", "Avg_Reviews"]

rows = []
for col in numeric_cols:
    s = games[col]
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = games.loc[(s < low) | (s > high), "Title"].tolist()
    rows.append({"Variable": col, "Q1": q1, "Q3": q3, "Lower limit": low, "Upper limit": high,
                 "Outliers (n)": len(flagged),
                 "Games": ", ".join(flagged[:4]) + ("..." if len(flagged) > 4 else "")})

report = pd.DataFrame(rows).set_index("Variable")
display(Markdown("**IQR rule on the numeric variables of `games`**"))
display(report.round(1))

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, numeric_cols):
    sns.boxplot(y=games[col], ax=ax)
    ax.set_title(col)
    ax.set_ylabel("")
fig.suptitle("Boxplots of the numeric variables of games (64 games)")
plt.tight_layout()
plt.show()


🔍 **Findings**<br>

- `Avg_Reviews` (limits 7.8 to 10.3) and `Total_team` (limits −4.1 to 16.9): no outlier.
- `Relevant_males`: 1 game, *Red Dead Redemption 2* (17 male characters). `Relevant_no_males`: 1 game, *I Was a Teenage Exocolonist* (14 non-male characters). Both are large casts, not errors.
- `Percentage_non_male`: 3 games at 100%, *Journey*, *Tunic* and *Chicory* (no male character at all). Real values.
- `Female_team`: 5 games with 3 or 4 women in the team (*Exocolonist*, *Immortality*, *Fire Emblem: Three Houses*, *Celeste*, *Child of Light*). The teams are small, so a few women are enough to flag them.

- ⚠️ `Protagonist`: 12 games are flagged (2, 3 or 4 protagonists), but this is a **false alarm**. 52 of the 64 games have exactly 1 protagonist, so Q1 = Q3 = 1 and the IQR is 0: any other count is flagged. They are valid counts.
- ⚠️ The same happens with `Sexualization` in `characters`: Q1 = Q3 = 0, so the 34 characters with a score above 0 would all be flagged. They are the characters that Q1 is about, so we do not treat them as outliers.

⚖️ **Decision:** keep all the flagged values.<br>
**Why:** none of them look like data errors. They are real games (large casts, all-female casts, teams with several women), and with only 64 games, removing them would lose information. 


### Team_percentage (share of women in the development team)

🔍 **Findings**<br>
- Q1 = 0%, median = 0%, Q3 = 14.75%, so IQR = 14.75 and the upper fence is about 36.9%.
- 5 of the 64 games lie above it: 43%, 50%, 60% and two at 67%.
- The distribution is strongly right-skewed (skewness ≈ 1.9): 36 of the 64 games (56%) have exactly 0% women in the team, and the mean (10.7%) is far above the median (0%).

⚖️ **Decision (provisional):** keep the 5 outlier games.<br>
**Why:** they are not data errors. They are real games with unusually balanced teams, and they are the most interesting cases for the team-mix question. Removing them would also leave only 59 games in a dataset that is already small.

⚠️ **Caution:** because the box is squashed at 0, the IQR rule flags a lot relative to the typical value. The mean is misleading here, so we use the median.


## 4. Dubious data

### Columns Metacritic, Destructoid, IGN, GameSpot and Avg_Reviews
Column "Avg_Reviews" which is supposed to be the average of the 4 review scores ("Metacritic ", "Destructoid", "IGN", "GameSpot")
We could consider dropping the 4 review score columns if Avg_Reviews is reliable.

In [ ]:
# First checking how Avg_Reviews was computed.

scores = ["Metacritic ", "Destructoid", "IGN", "GameSpot"]   # note the trailing space in "Metacritic "
recomputed = games[scores].mean(axis=1).round(1)
print("Games where Avg_Reviews differs from the recomputed mean:",
      (recomputed != games["Avg_Reviews"]).sum())


In [ ]:
# checking the distribution of the differences

diff = (recomputed - games["Avg_Reviews"]).round(2)
mismatch = games.loc[diff != 0, ["Title"] + scores + ["Avg_Reviews"]].copy()
mismatch["recomputed"] = recomputed[diff != 0]
mismatch["diff"] = diff[diff != 0]
display(mismatch)

🔍 **Findings**<br>

8 games differ by 0.1, and in all 8 the exact mean ends in 5 (e.g. 9.45), so the source rounds up and Python rounds down. 

It is only a rounding difference, so `Avg_Reviews` is reliable. The other columns can be dropped in the cleaning section

### Sexualization table
Column "Total" from Sexialization table is supposed to be the average of the 4 other columns ("Sexualized_clothing", "Trophy", "Damsel in Distress", "Sexualized Cutscenes")
We should make sure "Total" is reliable before using it in the analyze.

In [ ]:
sex = tables["sexualization"]
chars = tables["characters"]
criteria = ["Sexualized_clothing", "Trophy", "Damsel in Distress", "Sexualized Cutscenes"]

#Every criterion must be 0 or 1
print("Values outside {0,1}:", (~sex[criteria].isin([0, 1])).sum().sum())

#Total must equal the sum of the four criteria
print("Total != sum of criteria:", (sex["Total"] != sex[criteria].sum(axis=1)).sum())

#characters.Sexualization must equal sexualization.Total
merged = chars.merge(sex, on="Id", how="outer", indicator=True)
print("Ids missing in one table:", (merged["_merge"] != "both").sum())
print("Sexualization != Total:", (merged["Sexualization"] != merged["Total"]).sum())

#Metadata rule: a child or infant is never a "damsel in distress"
display(merged.query("Age_range in ['Child', 'Infant'] and `Damsel in Distress` == 1")
              [["Name", "Game", "Age_range", "Damsel in Distress", "Total"]])

games_c = tables["games"]

🔍 **Findings**<br>

Checks 1 to 3 all give 0, so the tables are fully consistent. 

Check 4 finds 1 real inconsistency: the character Son (game PP, a Child) has Damsel_in_distress = 1, which contradicts the author's own rule. (Children are expected to require assistance and are therefore not considered as "Damsel in distress") 

It's 1 of 637 rows, so it won't change the results, but it should be cleaned.

### Age vs Age_range
The numeric ages must fall inside their category

In [ ]:
age = pd.to_numeric(chars["Age"], errors="coerce")
labels = ["Infant", "Child", "Teenager", "Young adult", "Adult", "Middle-aged", "Elderly"]
expected = pd.cut(age, bins=[0, 6, 15, 18, 25, 40, 65, np.inf], right=False, labels=labels).astype(str)
print("Age outside its Age_range:", (age.notnull() & (expected != chars["Age_range"])).sum())

### Team_percentage 
Team_percentage must match Female_team / Total_team

In [ ]:
pct = games_c["Team_percentage"].str.rstrip("%").astype(float)
print("Team_percentage off by more than 1 point:",
      ((pct - games_c["female_team"] / games_c["Total_team"] * 100).abs() > 1).sum())

# 🧹 Data Cleaning

> We turn the raw tables into clean, analysis-ready ones: fixing types, filling or dropping gaps, removing columns we don't need and giving columns consistent names.

---

### 🎯 Goal
Apply the decisions taken in *Exploración*, so the analysis starts from tidy tables. Every change is explained (what we did and why) and the raw tables stay untouched.

### 📥 Input
The three tables from *Carga* (`games`, `characters`, `sexualization`) and the problems found in *Exploración* (empty column, gaps in `Series`, text values in numeric columns, inconsistent names).

### 🛠️ What we do
1. **Remove empty or useless columns:** `Unnamed: 27` (empty) and the columns no question uses (`Sub-genre`, `Publisher`, `Platform`, `PEGI`, `Criteria`), and `Age` from `characters` (duplicated by `Age_range`).
2. **Keep one review score:** keep `Avg_Reviews` and drop the four individual scores (`Metacritic`, `Destructoid`, `IGN`, `GameSpot`), once we verified it is reliable.
3. **Fill gaps with meaning:** `Series` → `"Standalone"` (a missing value means "no series").
4. **Rename columns** to one consistent style (`Female_team`, `Damsel_in_distress`, `Sexualized_cutscenes`).
5. **Fix data types:** `Release` text (`Nov-13`) → date and year; `Percentage_non_male` and `Team_percentage` (`"11%"`) → numbers; `Age` (numbers and words) → numeric where possible.
6. **Create new variables** (at least 3), for example `Release_year`, a female-lead flag, and groups of countries.
7. **Check the result:** `info()` on the clean tables, and no missing value left unexplained.

### 🔎 Rules we follow
| Situation | Treatment |
|---|---|
| A blank means "does not apply" (e.g. no series) | Fill with a label |
| A numeric value is truly unknown | Leave as `NaN`, never a text label |
| A column is empty or not used by any question | Drop it, with the reason |
| Two columns hold the same information | Keep one, after checking they match |

### 📝 Decisions
Each decision is written next to the code that applies it, as **what we did + why**.

### ✅ Result
*(Fill in after running the code.)*
- Clean tables: `games_clean`, `characters_clean`, `sexualization_clean`.
- Columns before → after, per table.

Notes:
- Steps 5 and 6 haven't been done in your notebook yet (the types and the new variables are tasks #3 and #14). Remove them from the list for now if you'd rather document only what is finished, or keep them as the plan and fill them in later.
- The "Rules we follow" table is only a summary of the principles you already applied. Delete it if you prefer the section shorter.
- Your current notebook does part of this work (the drops, the Series fill) inside Exploración. If you want the structure to match, move those cells under this Limpieza heading. Then Exploración only finds problems, and Limpieza only fixes them.

## 1. Remove empty or useless columns:

### Unnamed 

⚖️ **Decision:** drop `Unnamed: 27` from `games`. <br>
    **Why:** the column is completely empty. It comes from a trailing comma at the end of every row in the CSV, so it holds no information.

In [ ]:
tables["games"] = drop_empty_games_columns(tables["games"])
print(tables["games"].shape)

### Useless variables

⚖️ **Decision:** drop `Sub-genre`, `Publisher`, `Platform`, `PEGI` and `Criteria`.<br>
    **Why:** none of our three questions or their angles (time, team mix, country) uses them.

In [ ]:
tables["games"] = drop_unused_games_columns(tables["games"])
print(tables["games"].columns.tolist())

### Age from Characters table

⚖️ **Decision:** drop `Age` from `characters`. `Age_range` is the column we keep.<br>

**Why:** `Age` mixes exact numbers (150 characters) with words (487 characters). The words are identical to `Age_range`, and every numeric age falls inside its `Age_range` as defined in the metadata. So `Age_range` already holds the same information, complete for all 637 characters and with no missing values. None of our questions needs the exact age.

In [ ]:
characters = tables["characters"]

# Evidence before dropping: the words in Age equal Age_range, and Age_range has no gaps
age_num = pd.to_numeric(characters["Age"], errors="coerce")
words = characters.loc[age_num.isnull()]
print("Word values in Age that differ from Age_range:", (words["Age"] != words["Age_range"]).sum())
print("Missing Age_range:", characters["Age_range"].isnull().sum())

tables["characters"] = drop_age(characters)
print(tables["characters"].columns.tolist())


## 2. Keep one review score

⚖️ **Decision:** drop `Metacritic`, `Destructoid`, `IGN`, and `GameSpot`. Keep `Avg_Reviews`<br>
    **Why:** `Avg_Reviews` is the average of the four scores and has no gaps, while the individual scores have missing values. 
    
    `Avg_Reviews` has been found reliable in previous investigation.

In [ ]:
tables["games"] = drop_review_sites(tables["games"])
print(tables["games"].columns.tolist())

## 3. Fill gaps with meaning

⚖️ **Decision: fill the missing values of `Series` with `"Standalone"`.**<br>

    **Why:** a missing value here does not mean "unknown", it means the game belongs to no series. A label keeps these 37 games (58%) in `groupby` and charts, which ignore NaN by default.

In [ ]:
# Series: a missing value means the game belongs to no series, so we label it
tables["games"] = fill_series(tables["games"])
games = tables["games"]

# Check: no missing values left in those columns
print("Missing in Series:", games["Series"].isnull().sum())
print(games["Series"].value_counts().head(), "\n")

## 4. Rename columns

⚖️ **Decision:** Renaming columns whose name has a space, a hyphen, or does not start with a capital letter .<br>

    **Why:** Having a consistent naming convention. No other renaming seems necessary.

In [ ]:
# Finding columns whose name has a space, a hyphen, or does not start with a capital letter

for name, d in tables.items():
    bad = [c for c in d.columns if c != c.strip() or " " in c or "-" in c or not c[0].isupper()]
    print(f"{name}: {bad if bad else 'names OK'}")

# The rename mapping lives in src/cleaning.py (RENAME_MAP)
tables = rename_columns(tables)

for name, d in tables.items():
    print(f"{name}: {d.columns.tolist()}")

## 5. Fix Data Type

### Release from Games table

⚖️ **Decision:** convert `Release` from text (`"Nov-13"`) to a date, and create `Release_year`.<br>

**Why:** the time angle (A) of Q1–Q3 needs the year, and text can't be sorted or plotted over time. The months are English abbreviations, so we map them with an explicit English dictionary instead of relying on the computer's language settings.

In [ ]:
tables["games"] = fix_release_date(tables["games"])
games = tables["games"]

# Check: no missing date, range is 2012-2022
print("Missing dates:", games["Release"].isnull().sum())
print(games["Release"].min(), "->", games["Release"].max())

### `Percentage_non_male` and `Team_percentage` from Games table

⚖️ **Decision:** convert `Percentage_non_male` and `Team_percentage` from text (`"11%"`) to numbers (0-100).<br>

**Why:** a percentage stored as text can't be averaged, plotted or compared. We keep the 0-100 scale so values read as percentages.

In [ ]:
tables["games"] = fix_percentages(tables["games"])
print(tables["games"][["Percentage_non_male", "Team_percentage"]].dtypes)

In [ ]:
# Final check of the types
tables["games"].info()

### The Son Problem
Inconsistency found in the exploration phase. 
The character Son (game PP, a Child) has Damsel_in_distress = 1, which contradicts the author's own rule. (Children are expected to require assistance and are therefore not considered as "Damsel in distress") 

⚖️ **Decision:** Change this data from 1 to 0 and update his `Total` and `Sexualization` accordingly.<br>

**Why:** it is a clear inconsistency in conflict with autor's own rule. Additionnally, this error gives him a sexualization score of 1. In the data, only 5 men have a score above 0, and he is one of them. So the male count in relationship 1 is 5 now and would be 4 after the fix. The change is small, but it affects the main comparison.

In [ ]:
sex = tables["sexualization"]

son = sex["Id"] == "PP_Son"
print("Before:", sex.loc[son, ["Id", "Damsel_in_distress", "Total"]].to_dict("records"))

# Correct the flag, recompute Total, and copy it to characters.Sexualization
tables["sexualization"], tables["characters"] = fix_son_damsel_flag(tables["sexualization"], tables["characters"])

sex = tables["sexualization"]
print("After: ", sex.loc[sex["Id"] == "PP_Son", ["Id", "Damsel_in_distress", "Total"]].to_dict("records"))

In [ ]:
# check

merged = tables["characters"].merge(tables["sexualization"], on="Id")

print("Sexualization != Total:", (merged["Sexualization"] != merged["Total"]).sum())
kids = merged[merged["Age_range"].isin(["Infant", "Child"]) & (merged["Damsel_in_distress"] == 1)]
print("Children still flagged as damsel:", len(kids))



## 6. Outliers

⚖️ **Decision:** no outlier is removed or capped.<br>
    **Why:** the flagged values found in *Exploración* (section 3) are real games, not data errors, and removing them would shrink a dataset of only 64 games.


## 🛠️ 7. Feature engineering

### Is_sexualized in Characters table
Is_sexualized gives 34 True and 603 False.

⚖️ **Decision:** create `Is_sexualized` in `characters` (True if `Sexualization > 0`). <br>
**Why:** 603 of 637 characters score 0 and nobody scores 4, so the average score is misleading. A yes/no lets us compare the share of sexualized women and men. The unit is the character.

In [ ]:
tables["characters"] = add_is_sexualized(tables["characters"])
characters = tables["characters"]

print(characters["Is_sexualized"].value_counts())

### Has_female_team and Team_group in Games table
Team_group splits the 64 games into 36 (0%), 20 (1–25%) and 8 (>25%).

⚖️ **Decision:** create `Has_female_team` in `games` (True if `Team_percentage > 0`). <br>
**Why:** `Team_percentage` is 0% in 36 of 64 games and the median is 0%, so averaging or correlating the raw percentage is misleading. Comparing games with and without women in the team is more robust. The unit is the game.

⚖️ **Decision:** create `Team_group` in `games`, splitting `Team_percentage` into three groups: **0%**, **1-25%** and **>25%**. <br>
**Why:** `Has_female_team` hides how much women are involved. A split in three keeps the 0% group on its own, because it is more than half of the games, and separates small from larger shares. The cut at 25% is a choice, not a rule from the data, and the last group is small (8 games). The unit is the game.

In [ ]:
tables["games"] = add_has_female_team(tables["games"])
games = tables["games"]

print(games["Has_female_team"].value_counts())

In [ ]:
tables["games"] = add_team_group(tables["games"])
games = tables["games"]

print(games["Team_group"].value_counts(sort=False))

### Role_tier in Characters table

⚖️ **Decision:** create `Role_tier` in `characters`, grouping `Relevance` into three ordered levels: **Lead** (`PA`), **Major** (`DA`, `MC`, `MA`) and **Minor** (`SK`, `SC`). <br>
**Why:** "lead" can mean only the protagonist or all the important characters, and the metadata does not say how the author separates `PA` from `MC`. Three tiers let us report both the strict and the broad reading. Note that `Major` includes main antagonists (`MA`) and that `MC` is a very large group (202 characters). The unit is the character.

In [ ]:
tables["characters"] = add_role_tier(tables["characters"])
characters = tables["characters"]

print("Missing:", characters["Role_tier"].isna().sum())
display(Markdown("**Relevance code → Role_tier (number of characters)**"))
display(pd.crosstab(characters["Relevance"], characters["Role_tier"]))

### Share_women in Games table
Share_women is computed over all characters of the game, Non-binary and Custom included.

⚖️ **Decision:** create `Share_women` in `games`: the percentage of a game's characters who are women. <br>
**Why:** angle B compares games, but gender is recorded per character. This brings the character-level information to the game level. It is computed over all the game's characters, Non-binary and Custom included, and some games have few characters, so it can jump in steps. The unit is the game.

In [ ]:
tables["games"] = add_share_women(tables["games"], tables["characters"])
games = tables["games"]

print("Missing:", games["Share_women"].isna().sum())

### Share_sexualized in Games table
Share_sexualized is the percentage of a game's characters with a sexualization score above 0.

⚖️ **Decision:** create `Share_sexualized` in `games`: the percentage of a game's characters with a sexualization score above 0. <br>
**Why:** the sexualization data is per character, but the correlation compares games. Averaging `Is_sexualized` per game brings it to the game level. It is a share, so games with few characters can jump in steps. It is also very skewed: 45 of 64 games have no sexualized character at all. The unit is the game.

In [ ]:
tables["games"] = add_share_sexualized(tables["games"], tables["characters"])
games = tables["games"]

print("Missing:", games["Share_sexualized"].isna().sum())
print((games["Share_sexualized"] == 0).sum(), "of", len(games), "games have no sexualized character")

### Region in Games table

⚖️ **Decision:** create `Region` in `games`, grouping `Country` into four levels: **USA**, **Japan**, **Canada** and **Rest of world** (Europe + Australia). <br>
**Why:** `Country` has 13 values and most have only 1 to 4 games (USA 24, JPN 16, CAN 6, GBR 4, then 9 countries with 1 to 3 games each). Comparing them one by one would be noise, so the small ones are grouped. The unit is the game.

In [ ]:
tables["games"] = add_region(tables["games"])
games = tables["games"]

print("Missing:", games["Region"].isna().sum())
display(Markdown("**Number of games per Region**"))
display(games["Region"].value_counts().to_frame("Games"))

### Normalization: shares instead of counts

⚖️ **Decision:** compare games with **percentages** (`Share_women`, `Share_sexualized`, `Team_percentage`) instead of raw counts. <br>
**Why:** games have between 2 and 27 characters (median 9). A raw count of women mostly measures the size of the cast: its correlation with the number of characters is 0.84, against 0.17 for the share. With counts, the games with the most women would simply be the games with the most characters. Dividing by the cast size puts a 3-character game and a 27-character game on the same scale. <br>
**Limit:** a share jumps in big steps in small games (1 woman out of 3 = 33%), so we always show `n` next to it.

In [ ]:
display(Markdown("**Correlation with the number of characters per game: count vs share of women**"))
n_chars = characters.groupby("Game").size()
women = (characters["Gender"] == "Female").groupby(characters["Game"]).sum()
print("count of women :", round(n_chars.corr(women), 2))
print("share of women :", round(n_chars.corr(games.set_index("Game_Id")["Share_women"]), 2))

In [ ]:
# Check

display(Markdown("**New variables, first 10 games**"))
display(games[["Title", "Team_percentage", "Has_female_team", "Team_group", "Share_women"]].head(10))
print(games["Team_group"].value_counts(sort=False))
print("Missing Share_women:", games["Share_women"].isna().sum())

### Same result in one call

The steps above call the functions of `src/cleaning.py` one at a time, so each decision sits next to its code. `clean_all` chains them all. It is what the dashboard (`app.py`) and later the pipeline use, so we check that both routes give exactly the same tables.

In [ ]:
cleaned = clean_all(read_raw_tables(path))

for name in tables:
    pd.testing.assert_frame_equal(tables[name], cleaned[name])   # raises an error if anything differs
    print(f"{name}: identical ({tables[name].shape[0]} rows, {tables[name].shape[1]} columns)")

# 📊 Análisis

> With clean tables, we study the variables alone (univariate) and in pairs (bivariate) to answer Q1–Q3.

---

### 🎯 Goal
Describe each variable we need, then compare groups (mainly women vs men) to answer our three questions.

### 📥 Input
The clean tables from *Limpieza* (`games`, `characters`, `sexualization`).

### 🛠️ What we do
1. **Univariate analysis** (at least 3 variables):
   - categorical (e.g. `Gender`, `Relevance`) → bar chart of `value_counts()`
   - numeric (e.g. `Team_percentage`) → histogram + boxplot
2. **Bivariate analysis** (at least 3 relationships):
   - category vs category → `pd.crosstab(..., normalize="index") * 100` + grouped bars (e.g. `Gender` × `Sexualization`)
   - number vs category → boxplot by group
3. Show `n` next to every percentage: the dataset is small.

### 📊 Charts checklist
Every chart has a title, named axes, and a legend when it has more than one series.


In [ ]:
# Role names, defined once and reused by the role analyses below
role_labels = {
    "PA": "Protagonist", "DA": "Deuteragonist", "MA": "Main antagonist",
    "MC": "Main character", "SK": "Sidekick", "SC": "Secondary character",
}
role_names = list(role_labels.values())      # ordered from most to least central to the story
tier_names = ["Lead", "Major", "Minor"]      # Role_tier levels, see Feature engineering


## 1. Univariate analysis

### Gender Variable from Characters table

In [ ]:
characters = tables["characters"]

# 1. The numbers
counts = characters["Gender"].value_counts()
summary = pd.DataFrame({
    "n": counts,
    "percent": (counts / len(characters) * 100).round(1),
})
display(summary)

# 2. Bar chart
fig, ax = plt.subplots()
sns.barplot(x=counts.index, y=counts.values, ax=ax)
ax.set_title("Characters by gender (637 characters)")
ax.set_xlabel("Gender")
ax.set_ylabel("Number of characters")

# write "n (percent)" above each bar
for i, n in enumerate(counts.values):
    ax.text(i, n + 5, f"{n} ({n / len(characters) * 100:.1f}%)", ha="center")

plt.show()

🔍 **Findings**<br>

- men are about 62% of characters (393 of 637) and women about 32% (203). So there are roughly two men for every woman.
- Non-binary and Custom are smaller groups (33 and 8) 

### Team percentage from Games table

In [ ]:
games = tables["games"]

team_pct = games["Team_percentage"]

# 1. The numbers
print(team_pct.describe())
print("Median:", team_pct.median())
print("Skewness:", round(team_pct.skew(), 2))
print("Games with 0%:", (team_pct == 0).sum(), f"({(team_pct == 0).mean() * 100:.0f}%)")

# 2. Histogram
fig, ax = plt.subplots()
sns.histplot(team_pct, bins=15, ax=ax)
ax.axvline(team_pct.mean(), color="red", linestyle="--", label=f"Mean ({team_pct.mean():.1f}%)")
ax.axvline(team_pct.median(), color="green", linestyle="-", label=f"Median ({team_pct.median():.1f}%)")
ax.set_title("Share of women in the development team (64 games)")
ax.set_xlabel("Team_percentage (%)")
ax.set_ylabel("Number of games")
ax.legend()
plt.show()

# 3. Boxplot
fig, ax = plt.subplots(figsize=(10, 3))
sns.boxplot(x=team_pct, ax=ax)
ax.set_title("Boxplot of the share of women in the development team")
ax.set_xlabel("Team_percentage (%)")
plt.show()

🔍 **Findings**<br>

- Mean vs. median: the mean (10.7) is well above the median (0). This gap is the sign of a skewed distribution, and a few high values are pulling the mean up.

- Histogram: one tall bar at 0 and a long tail to the right.

- Boxplot: the box is squashed at 0 because 25% and 50% are both 0 and the 75% mark is only 14.75. The dots on the right (43, 50, 60, 67, 67) are flagged as outliers by the IQR rule.

- The typical game in this dataset has no women in its core team, and the mean misleads here, so we will prefer the median.

### Roles from characters

In [ ]:
characters = tables["characters"]

# 1. The numbers
counts = characters["Relevance"].value_counts()
summary = pd.DataFrame({
    "role": counts.index.map(role_labels),
    "n": counts.values,
    "percent": (counts.values / len(characters) * 100).round(1),
}, index=counts.index)
display(summary)

# 2. Bar chart (sorted from most to least frequent)
fig, ax = plt.subplots()
sns.barplot(x=counts.values, y=counts.index.map(role_labels), ax=ax)
ax.set_title("Characters by role in the story (637 characters)")
ax.set_xlabel("Number of characters")
ax.set_ylabel("Role (Relevance)")

for i, n in enumerate(counts.values):
    ax.text(n + 3, i, f"{n} ({n / len(characters) * 100:.1f}%)", va="center")

plt.show()
# 3. The three role tiers (Role_tier, see Feature engineering)
tier_counts = characters["Role_tier"].value_counts().reindex(tier_names)
tier_summary = pd.DataFrame({
    "n": tier_counts,
    "percent": (tier_counts / len(characters) * 100).round(1),
})
display(Markdown("**Characters by role tier**"))
display(tier_summary)

fig, ax = plt.subplots()
sns.barplot(x=tier_names, y=tier_counts.values, ax=ax)
ax.set_title("Characters by role tier (637 characters)")
ax.set_xlabel("Role tier")
ax.set_ylabel("Number of characters")

for i, n in enumerate(tier_counts.values):
    ax.text(i, n + 5, f"{n} ({n / len(characters) * 100:.1f}%)", ha="center")

plt.show()


🔍 **Findings**<br>

- The majority are background: secondary characters are about 42% (266 of 637). Main characters are another 32%.

- Protagonists are rare: 80 characters, about 13%. 


## 2. Bivariate analysis

### Gender & Sexualization

In [ ]:
characters = tables["characters"]

# Women and men only: Non-binary (33) and Custom (8) are too small to compare (see notes-for-later)
subset = characters[characters["Gender"].isin(["Female", "Male"])].copy()

# 1. The numbers: counts and row percentages
counts = pd.crosstab(subset["Gender"], subset["Is_sexualized"])
percent = pd.crosstab(subset["Gender"], subset["Is_sexualized"], normalize="index") * 100
display(counts)
display(percent.round(1))

# 2. Bar chart: share of sexualized characters per gender
rates = percent[True]

fig, ax = plt.subplots()
sns.barplot(x=rates.index, y=rates.values, ax=ax)
ax.set_title("Share of sexualized characters, women vs men")
ax.set_xlabel("Gender")
ax.set_ylabel("Characters with a score above 0 (%)")

for i, g in enumerate(rates.index):
    ax.text(i, rates[g] + 0.3, f"{counts.loc[g, True]}/{counts.loc[g].sum()} ({rates[g]:.1f}%)", ha="center")

plt.show()

🔍 **Findings**<br>

- 29 of 203 women (14.3%) have a score above 0, against 4 of 393 men (1%). That is roughly 14 times higher.



### Gender & Role

In [ ]:
characters = tables["characters"]

# Women and men only (Non-binary and Custom are too small, see notes-for-later)
subset = characters[characters["Gender"].isin(["Female", "Male"])].copy()

subset["Role"] = subset["Relevance"].map(role_labels)

# 1. The numbers
counts = pd.crosstab(subset["Gender"], subset["Role"])[role_names]
within_gender = (pd.crosstab(subset["Gender"], subset["Role"], normalize="index") * 100)[role_names]
within_role = (pd.crosstab(subset["Gender"], subset["Role"], normalize="columns") * 100)[role_names]

display(Markdown("**Number of characters (n)**"))
display(counts)                  
display(Markdown("**% of each gender holding each role** (each row adds up to 100%)"))
display(within_gender.round(1)) 
display(Markdown("**% of each role held by each gender** (each column adds up to 100%)"))
display(within_role.round(1))

# 2. Grouped bar chart (within each gender)
plot_data = within_gender.reset_index().melt(id_vars="Gender", var_name="Role", value_name="Percent")
n = subset["Gender"].value_counts()
plot_data["Gender"] = plot_data["Gender"].map(lambda g: f"{g} (n={n[g]})")

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=plot_data, x="Role", y="Percent", hue="Gender", ax=ax)
ax.set_title("Roles held by women and men (% of each gender)")
ax.set_xlabel("Role in the story")
ax.set_ylabel("Share of the gender's characters (%)")
plt.xticks(rotation=20)
plt.show()

# 3. The same comparison with the three role tiers (Role_tier)
tier_n = pd.crosstab(subset["Gender"], subset["Role_tier"])
tier_within_gender = pd.crosstab(subset["Gender"], subset["Role_tier"], normalize="index") * 100
tier_within_tier = pd.crosstab(subset["Gender"], subset["Role_tier"], normalize="columns") * 100

display(Markdown("**Number of characters by role tier (n)**"))
display(tier_n)
display(Markdown("**% of each gender in each role tier** (each row adds up to 100%)"))
display(tier_within_gender.round(1))
display(Markdown("**% of each role tier held by each gender** (each column adds up to 100%)"))
display(tier_within_tier.round(1))

# 4. Share of women in each tier, against their share among all women and men
women_overall = (subset["Gender"] == "Female").mean() * 100
tier_totals = tier_n.sum()

fig, ax = plt.subplots()
sns.barplot(x=[f"{t} (n={tier_totals[t]})" for t in tier_names],
            y=tier_within_tier.loc["Female", tier_names].values, ax=ax)
ax.axhline(women_overall, color="red", linestyle="--", label=f"Women overall ({women_overall:.1f}%)")
ax.set_title("Share of women in each role tier")
ax.set_xlabel("Role tier")
ax.set_ylabel("Women (% of the tier's women and men)")
ax.legend()
plt.show()


🔍 **Findings**<br>

- Women are about 32% of the characters, but only 26% of the protagonists. Among main antagonists, only 22% are women. Those two are the roles where women are the least present.

- Women are over-represented as main characters (39% of the role, against 32% overall) and slightly over-represented as deuteragonists and sidekicks.

- Within each gender, the profiles are close. Women have a larger share of main characters (38% against 31%), and men have a larger share of protagonists and main antagonists.

- Several groups are small: deuteragonist (18) and sidekick (13) are too few to draw conclusions. 

### Gender and Side 

In [ ]:
characters = tables["characters"]

# Women and men only (Non-binary and Custom are too small, see notes-for-later)
subset = characters[characters["Gender"].isin(["Female", "Male"])].copy()

side_labels = {"P": "Protagonist's side", "A": "Antagonist", "B": "Both sides"}
side_names = list(side_labels.values())
subset["Side_label"] = subset["Side"].map(side_labels)

# 1. The numbers
counts = pd.crosstab(subset["Gender"], subset["Side_label"])[side_names]
within_gender = (pd.crosstab(subset["Gender"], subset["Side_label"], normalize="index") * 100)[side_names]
within_side = (pd.crosstab(subset["Gender"], subset["Side_label"], normalize="columns") * 100)[side_names]

display(Markdown("**Number of characters (n)**"))
display(counts)

display(Markdown("**% of each gender on each side** (each row adds up to 100%)"))
display(within_gender.round(1))

display(Markdown("**% of each side held by each gender** (each column adds up to 100%)"))
display(within_side.round(1))

# 2. Grouped bar chart (within each gender)
plot_data = within_gender.reset_index().melt(id_vars="Gender", var_name="Side", value_name="Percent")
n = subset["Gender"].value_counts()
plot_data["Gender"] = plot_data["Gender"].map(lambda g: f"{g} (n={n[g]})")

fig, ax = plt.subplots()
sns.barplot(data=plot_data, x="Side", y="Percent", hue="Gender", ax=ax)
ax.set_title("Side taken by women and men (% of each gender)")
ax.set_xlabel("Side in the story")
ax.set_ylabel("Share of the gender's characters (%)")
plt.show()

🔍 **Findings**<br>

- Women are mostly on the protagonist's side: 75% of them, against 61% of men.

-  Women are rarely antagonists: about 10% of women, against 28% of men. They are only 15.5% of all antagonists, although they are 32% of the characters. 

-  The "Both sides" group is a bit larger for women (15% against 11%), and women are 41% of it. It is small (73 characters in total), we need to look at this number with caution.

## 3. Group Summaries

### ❔ Does the sexualization gap depend on the role?
Analyze through a pivot table (groupby) crossing the data of Gender, Sexualization and Role.
Each cell of the grid answers: "among women or men who with this role, what share are sexualized?"

In [ ]:
characters = tables["characters"]

# Women and men only (Non-binary and Custom are too small, see notes-for-later)
subset = characters[characters["Gender"].isin(["Female", "Male"])].copy()
subset["Role"] = subset["Relevance"].map(role_labels)

def sexualized_counts(index_col, order):
    """n, number sexualized and % sexualized, for each group of index_col and each gender."""
    def pivot(aggfunc):
        return pd.pivot_table(subset, values="Is_sexualized", index=index_col,
                              columns="Gender", aggfunc=aggfunc,
                              observed=False).reindex(order)
    return pivot("count"), pivot("sum"), pivot("mean") * 100

def as_text(n, sexualized, pct):
    """The compact 'x / n (p%)' version for reading."""
    return (sexualized.astype(int).astype(str) + " / " + n.astype(int).astype(str)
            + " (" + pct.round(1).astype(str) + "%)")

# By role (n, sexualized and pct stay available for the chart below)
n, sexualized, pct = sexualized_counts("Role", role_names)
display(Markdown("**Sexualized characters / n (%), by role and gender**"))
display(as_text(n, sexualized, pct))

# By role tier
display(Markdown("**Sexualized characters / n (%), by role tier and gender**"))
display(as_text(*sexualized_counts("Role_tier", tier_names)))

role_plot = pd.concat({
    "Sexualized": sexualized.astype(int), "n": n.astype(int), "Percent": pct.round(1),
}, axis=1).stack(level=1).reset_index()
role_plot.columns = ["Role", "Gender", "Sexualized", "n", "Percent"]

fig = px.bar(role_plot, x="Role", y="Percent", color="Gender", barmode="group",
             custom_data=["Sexualized", "n"],
             title="Sexualized characters by role, women vs men (% of each gender in the role)",
             labels={"Percent": "Characters with a score above 0 (%)", "Role": "Role in the story"})
fig.update_traces(hovertemplate="<b>%{x}</b><br>%{fullData.name}: %{y:.1f}%"
                                "<br>%{customdata[0]} sexualized / %{customdata[1]} characters<extra></extra>")
fig.update_xaxes(categoryorder="array", categoryarray=role_names)
fig.show()


🔍 **Findings**<br>

- The gap holds across the roles with enough characters. Among protagonists, main characters and secondary characters, women are at 12–14% and men at 1–2%. That points to the cause being gender, not role.

- Main antagonists are at 0% for both. The 11 women who are main antagonists aren't sexualized in this dataset.

- The 50% for female deuteragonists is 4 of 8 women. Sidekicks have 6 women and 7 men. Men have only 4 sexualized characters in total, one per role, so the male percentages are fragile.

### ❔ Which of the four criteria of sexualization drives the gap ?

In [ ]:
criteria = ["Sexualized_clothing", "Trophy", "Damsel_in_distress", "Sexualized_cutscenes"]
criteria_labels = {"Sexualized_clothing": "Sexualized clothing", "Trophy": "Trophy",
                   "Damsel_in_distress": "Damsel in distress", "Sexualized_cutscenes": "Sexualized cutscenes"}

data = tables["characters"][["Id", "Gender"]].merge(tables["sexualization"], on="Id")
data = data[data["Gender"].isin(["Female", "Male"])]

n = data["Gender"].value_counts()
flagged = data.groupby("Gender")[criteria].sum()
pct = data.groupby("Gender")[criteria].mean() * 100

summary = flagged.astype(int).astype(str) + " (" + pct.round(1).astype(str) + "%)"
summary = summary.rename(columns=criteria_labels)
summary.insert(0, "Characters", n)

display(Markdown("**Characters flagged on each criterion, by gender: n (% of the gender)**"))
display(summary)

plot_data = pct.rename(columns=criteria_labels).reset_index().melt(
    id_vars="Gender", var_name="Criterion", value_name="Percent")
plot_data["Gender"] = plot_data["Gender"].map(lambda g: f"{g} (n={n[g]})")

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=plot_data, x="Criterion", y="Percent", hue="Gender", ax=ax)
ax.set_title("Sexualization criteria, women vs men (% of each gender)")
ax.set_xlabel("Criterion")
ax.set_ylabel("Share of the gender's characters (%)")
plt.tight_layout()
plt.show()

🔍 **Findings**<br>

- Women are above men on all four criteria. The biggest gaps are in sexualized clothing and the damsel in distress role. A character can be flagged on several criteria at once, so the counts don't add up to the number of sexualized women.

### ❔ What is the Gender repartition in playable and romantic interest ?

In [ ]:
characters = tables["characters"]
subset = characters[characters["Gender"].isin(["Female", "Male"])].copy()
subset["Playable_flag"] = subset["Playable"] == 1
subset["Romantic_flag"] = subset["Romantic_Interest"].isin(["Yes", "Opt"])

n = subset["Gender"].value_counts()
playable = subset.groupby("Gender")["Playable_flag"].agg(["sum", "mean"])
romantic = subset.groupby("Gender")["Romantic_flag"].agg(["sum", "mean"])
romantic_detail = pd.crosstab(subset["Gender"], subset["Romantic_Interest"])

summary2 = pd.DataFrame({
    "Characters": n,
    "Playable": playable["sum"].astype(int).astype(str) + " ("
        + (playable["mean"] * 100).round(1).astype(str) + "%)",
    "Romantic interest (Yes or Opt)": romantic["sum"].astype(int).astype(str) + " ("
        + (romantic["mean"] * 100).round(1).astype(str) + "%)",
})

display(Markdown("**Playable characters and romantic interests, by gender: n (% of the gender)**"))
display(summary2)
display(Markdown("**Romantic interest in detail** (`Yes` = unavoidable partner, `Opt` = player's choice)"))
display(romantic_detail[["No", "Opt", "Yes"]])

plot2 = pd.DataFrame({"Playable": playable["mean"] * 100, "Romantic interest": romantic["mean"] * 100})
plot2 = plot2.reset_index().melt(id_vars="Gender", var_name="Measure", value_name="Percent")
plot2["Gender"] = plot2["Gender"].map(lambda g: f"{g} (n={n[g]})")

fig, ax = plt.subplots()
sns.barplot(data=plot2, x="Measure", y="Percent", hue="Gender", ax=ax)
ax.set_title("Playable characters and romantic interests, women vs men")
ax.set_xlabel("Measure")
ax.set_ylabel("Share of the gender's characters (%)")
plt.tight_layout()
plt.show()

🔍 **Findings**<br>

- Women are playable almost as often as men (16% against 18%), so being playable isn't where they're under-represented. 
But women are about 3.7 times more often a romantic interest (15.3% against 4.1%). 


### ❔ Do games made by teams with at least one woman have more female characters ? Are they less sexualized ?
Analyze through a groupby("Game") and agg(name=(column, function)).
We are grouping characters by game first, so that games with many characters don't weigh more.

In [ ]:
characters = tables["characters"]
games = tables["games"]

# One row per game. The team variables and Share_women come from Feature engineering;
# only the number of characters and of sexualized characters is counted here.
per_game = games[["Game_Id", "Title", "Team_percentage", "Has_female_team", "Team_group", "Share_women"]].copy()
per_game["n_characters"] = per_game["Game_Id"].map(characters.groupby("Game")["Id"].count())
per_game["n_sexualized"] = per_game["Game_Id"].map(characters.groupby("Game")["Is_sexualized"].sum())

display(Markdown("**One row per game**"))
display(per_game.head())

def team_summary(group_col):
    by_group = per_game.groupby(group_col, observed=True).agg(
        n_games=("Game_Id", "count"),
        n_characters=("n_characters", "sum"),
        n_sexualized=("n_sexualized", "sum"),
        mean_share_women=("Share_women", "mean"),
        median_share_women=("Share_women", "median"),
    )
    return pd.DataFrame({
        "Games": by_group["n_games"],
        "Characters": by_group["n_characters"],
        "Women among characters (mean / median)":
            by_group["mean_share_women"].round(1).astype(str) + "% / "
            + by_group["median_share_women"].round(1).astype(str) + "%",
        "Sexualized characters": (by_group["n_sexualized"].astype(str) + " ("
            + (by_group["n_sexualized"] / by_group["n_characters"] * 100).round(1).astype(str) + " per 100)"),
    })

# Comparing the two groups: games with a female team vs games without a female team
summary = team_summary("Has_female_team")
summary.index = summary.index.map({False: "0% women in the team", True: "At least one woman"})
summary.index.name = "Team"

display(Markdown("**Games grouped by whether the team includes women**"))
display(summary)

# The same comparison with three groups (Team_group)
summary3 = team_summary("Team_group")
summary3.index.name = "Women in the team"

display(Markdown("**Games grouped by the share of women in the team**"))
display(summary3)

# Share of women among the characters, by team group
group_order = ["0%", "1-25%", ">25%"]
group_n = per_game["Team_group"].value_counts()

fig, ax = plt.subplots()
sns.boxplot(data=per_game, x="Team_group", y="Share_women", order=group_order, ax=ax)
ax.set_xticks(range(len(group_order)))
ax.set_xticklabels([f"{g} (n={group_n[g]})" for g in group_order])
ax.set_title("Share of women among the characters, by share of women in the team")
ax.set_xlabel("Women in the development team")
ax.set_ylabel("Women among the game's characters (%)")
plt.show()

per_game["Team"] = per_game["Has_female_team"].map({False: "0% women in the team", True: "At least one woman"})
team_n = per_game["Team"].value_counts()
per_game["Team_label"] = per_game["Team"].map(lambda t: f"{t} (n={team_n[t]} games)")

fig = px.box(per_game, x="Team_label", y="Share_women", points="all",
             hover_name="Title", hover_data={"Team_label": False, "Share_women": ":.1f",
                                             "n_characters": True, "Team_percentage": True},
             title="Share of female characters per game, by presence of women in the team",
             labels={"Share_women": "Women among the game's characters (%)", "Team_label": "Development team",
                     "n_characters": "Characters in the game", "Team_percentage": "Women in team (%)"})
fig.show()


🔍 **Findings**<br>

- Games with a woman in the team have more female characters: about 35% against 27% on average, and 40% against 30% for the median.

- Sexualization is slightly lower in those games (4.2 against 6.2 per 100 characters), but the numbers are small (11 against 23 sexualized characters). It should be taken lightly

### ❔ Do we see an evolution of the trends through time, between 2012 and 2022 ?

Using `groupby("Release_year")` and `agg`, we count, for each release year, all the characters, how many are women, and how many of those women are sexualized.
Here the unit is the character (not the game), so that the sexualization counts match the character counts.

In [ ]:
characters_year = tables["characters"].merge(
    tables["games"][["Game_Id", "Release_year"]], left_on="Game", right_on="Game_Id", how="left")
characters_year["Is_female"] = characters_year["Gender"] == "Female"

women_over_time = characters_year.groupby("Release_year").agg(
    Characters=("Id", "count"),
    Women=("Is_female", "sum"),
)
women_over_time["Women_sexualized"] = (
    characters_year[characters_year["Is_female"]].groupby("Release_year")["Is_sexualized"].sum())
women_over_time["Pct_female_characters"] = women_over_time["Women"] / women_over_time["Characters"] * 100
women_over_time["Pct_women_sexualized"] = women_over_time["Women_sexualized"] / women_over_time["Women"] * 100

summary_time = pd.DataFrame({
    "Characters": women_over_time["Characters"],
    "Female characters": women_over_time["Women"].astype(int).astype(str) + " ("
        + women_over_time["Pct_female_characters"].round(1).astype(str) + "%)",
    "Sexualized women": women_over_time["Women_sexualized"].astype(int).astype(str) + " / "
        + women_over_time["Women"].astype(int).astype(str) + " ("
        + women_over_time["Pct_women_sexualized"].round(1).astype(str) + "%)",
})

display(Markdown("**Female characters and their sexualization, by release year**"))
display(summary_time)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(women_over_time.index, women_over_time["Pct_female_characters"], marker="o",
        label="Female characters (% of characters)")
ax.plot(women_over_time.index, women_over_time["Pct_women_sexualized"], marker="s",
        label="Sexualized women (% of women)")
ax.set_title("Women in video games over time: representation and sexualization")
ax.set_xlabel("Release year")
ax.set_ylabel("Percentage (%)")
ax.set_xticks(women_over_time.index)
ax.set_ylim(0, 50)
ax.legend()
plt.tight_layout()
plt.show()

⚖️ **Decision:** smooth the yearly trend with a **3-year centered rolling window**, adding up the counts in the window before dividing (pooled), and keep the raw line next to it. <br>
**Why:** each year rests on only 12 to 38 women and 0 to 8 sexualized women, so one character moves a yearly percentage by several points. Pooling 3 years reduces that noise, and adding the counts (instead of averaging the yearly percentages) gives bigger years more weight. <br>
**Limit:** the first and last years use only 2 years. Neighbouring points share data, so the smoothed line shows a direction, not independent yearly values.

In [ ]:
window = women_over_time[["Characters", "Women", "Women_sexualized"]].rolling(3, center=True, min_periods=2).sum()
women_over_time["Pct_female_smooth"] = window["Women"] / window["Characters"] * 100
women_over_time["Pct_sexualized_smooth"] = window["Women_sexualized"] / window["Women"] * 100

display(Markdown("**Raw vs 3-year smoothed percentages, by release year**"))
display(women_over_time[["Pct_female_characters", "Pct_female_smooth",
                         "Pct_women_sexualized", "Pct_sexualized_smooth"]].round(1))

In [ ]:
trend = women_over_time.reset_index().rename(columns={"Release_year": "Year"})
FEMALE, SEXUALIZED = "Female characters (% of characters)", "Sexualized women (% of women)"

trend_plot = pd.concat([
    pd.DataFrame({"Year": trend["Year"], "Measure": FEMALE, "Version": "Raw (yearly)",
                  "Percent": trend["Pct_female_characters"].round(1),
                  "Count": trend["Women"], "Base": trend["Characters"]}),
    pd.DataFrame({"Year": trend["Year"], "Measure": FEMALE, "Version": "3-year smoothed",
                  "Percent": trend["Pct_female_smooth"].round(1),
                  "Count": window["Women"].values, "Base": window["Characters"].values}),
    pd.DataFrame({"Year": trend["Year"], "Measure": SEXUALIZED, "Version": "Raw (yearly)",
                  "Percent": trend["Pct_women_sexualized"].round(1),
                  "Count": trend["Women_sexualized"], "Base": trend["Women"]}),
    pd.DataFrame({"Year": trend["Year"], "Measure": SEXUALIZED, "Version": "3-year smoothed",
                  "Percent": trend["Pct_sexualized_smooth"].round(1),
                  "Count": window["Women_sexualized"].values, "Base": window["Women"].values}),
])

fig = px.line(trend_plot, x="Year", y="Percent", color="Measure", line_dash="Version",
              line_dash_map={"Raw (yearly)": "dot", "3-year smoothed": "solid"},
              markers=True, custom_data=["Count", "Base"],
              title="Women in video games over time: raw yearly values vs 3-year smoothing",
              labels={"Percent": "Percentage (%)", "Year": "Release year"})
for trace in fig.data:
    if "smoothed" in trace.name:
        trace.update(line_width=4)
    else:
        trace.update(line_width=1.5, opacity=0.5)
fig.update_traces(hovertemplate="<b>%{x}</b><br>%{y:.1f}%  (%{customdata[0]} of %{customdata[1]})<extra>%{fullData.name}</extra>")
fig.update_xaxes(dtick=1)
fig.update_yaxes(range=[0, 50])
fig.update_layout(hovermode="x unified")
fig.show()

🔍 **Findings**<br>

- The raw yearly line zigzags, but the 3-year smoothed line shows a slow rise in representation: the share of female characters goes from about 28% (2012) to 38% (2022). The 45% peak of 2021 (19 of 42 characters) is partly a one-year spike, since the smoothed value for that year is 37.6%.

- Sexualization trends down, and the smoothing makes it clearer. The share of sexualized women stays around 17–23% until 2016, then falls to about 5% in 2020–2022. The raw line says the same, with two years at 0% (2018 and 2021).

- The two lines separate after 2017: more women among the characters, and fewer of them sexualized.

- Each yearly point rests on 12 to 38 women, and on only 0 to 8 sexualized women, so a single character moves a yearly percentage by several points. The smoothed values pool 3 years, so neighbouring points share data (and the first and last years use only 2 years): they show a direction, not independent yearly facts. We did not test whether the rise is more than chance, so this is a tendency in the data, not a proven trend.

- These are top-selling or top-rated games, so this describes successful games, not the industry.

### ❔ Does the developer's country make a difference ?

Angle C of Q1 and Q2. We compare the four regions built in *Feature engineering* (`Region`). <br>
The unit is the **character** for the percentages (women among characters, sexualized women among women) and the **game** for the team median, so we say which one each table uses.

In [ ]:
characters = tables["characters"]
games = tables["games"]

chars_region = characters.assign(Region=characters["Game"].map(games.set_index("Game_Id")["Region"]))
chars_region["Is_female"] = chars_region["Gender"] == "Female"

by_region = chars_region.groupby("Region").agg(
    Games=("Game", "nunique"), Characters=("Id", "count"), Women=("Is_female", "sum"))
by_region["Sexualized_women"] = chars_region[chars_region["Is_female"]].groupby("Region")["Is_sexualized"].sum()
by_region["Pct_women"] = by_region["Women"] / by_region["Characters"] * 100
by_region["Pct_sexualized"] = by_region["Sexualized_women"] / by_region["Women"] * 100
by_region = by_region.loc[["USA", "Japan", "Canada", "Rest of world"]]

summary_region = pd.DataFrame({
    "Games": by_region["Games"],
    "Women": by_region["Women"].astype(int).astype(str) + " / " + by_region["Characters"].astype(str)
        + " (" + by_region["Pct_women"].round(1).astype(str) + "%)",
    "Sexualized women": by_region["Sexualized_women"].astype(int).astype(str) + " / "
        + by_region["Women"].astype(int).astype(str) + " (" + by_region["Pct_sexualized"].round(1).astype(str) + "%)",
})
display(Markdown("**Women and sexualized women by developer region (unit: character)**"))
display(summary_region)

display(Markdown("**Development team by developer region (unit: game)**"))
display(games.groupby("Region")["Team_percentage"].agg(Games="count", Median_team_pct="median", Mean_team_pct="mean")
        .loc[["USA", "Japan", "Canada", "Rest of world"]].round(1))

In [ ]:
plot = by_region.reset_index()
plot["Label"] = (plot["Region"] + "\n(" + plot["Games"].astype(str) + " games, "
                 + plot["Women"].astype(int).astype(str) + " women)")

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=plot, x="Label", y="Pct_sexualized", color="#c0504d", ax=ax)
for i, r in plot.iterrows():
    ax.text(i, r["Pct_sexualized"] + 0.5,
            f"{r['Pct_sexualized']:.1f}%\n({int(r['Sexualized_women'])} of {int(r['Women'])})", ha="center")
ax.set_title("Sexualized women by developer region")
ax.set_xlabel("Developer region")
ax.set_ylabel("Sexualized women (% of women)")
ax.set_ylim(0, 30)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import fisher_exact

jp, us = by_region.loc["Japan"], by_region.loc["USA"]
p_sex = fisher_exact([[jp["Sexualized_women"], jp["Women"] - jp["Sexualized_women"]],
                      [us["Sexualized_women"], us["Women"] - us["Sexualized_women"]]])[1]
p_rep = fisher_exact([[jp["Women"], jp["Characters"] - jp["Women"]],
                      [us["Women"], us["Characters"] - us["Women"]]])[1]
print(f"Japan vs USA, sexualized women: {jp['Pct_sexualized']:.1f}% vs {us['Pct_sexualized']:.1f}%  (Fisher p = {p_sex:.3f})")
print(f"Japan vs USA, share of women:   {jp['Pct_women']:.1f}% vs {us['Pct_women']:.1f}%  (Fisher p = {p_rep:.3f})")

🔍 **Findings**<br>

- Representation hardly depends on the country: women are 30% to 36% of the characters in every region, and Japan and the USA are almost the same (31.1% against 32.3%, Fisher p = 0.83).

- Sexualization does depend on it. In Japanese games 12 of 55 women (21.8%) are sexualized, against 6 of 81 (7.4%) in US games, about three times as often (Fisher p = 0.02). The Rest of world is close to Japan (8 of 42, 19.0%) and Canada is in between (3 of 25, 12.0%), but these groups are small.

- Teams: in most regions the median game has no woman in the team (0%). Canada is the exception (median 34%), but it rests on only 6 games.

- Caution: a few games weigh a lot (the only Polish game, The Witcher 3, supplies 4 of the 8 sexualized women of the Rest of world), and 12 of the 55 Japanese women is a small count. The p-value is a rough guide, not proof. These are top-selling or top-rated games, so this describes successful games, not the national industries.


## 4. Correlation

⚖️ **Decision:** compute the correlation at the game level (64 games) with **Spearman**, on seven variables. <br>
**Why:** `Team_percentage` and `Share_sexualized` are very skewed (36 and 45 of 64 games sit at 0), so Spearman, which works on ranks, is safer than Pearson. We leave out `Female_team` (it is the same information as `Team_percentage`, r = 0.84) and the character-count columns, which overlap with `Percentage_non_male` and `Share_women`. With 64 games, a correlation of about ±0.25 is borderline.

In [ ]:
games = tables["games"]

cols = ["Team_percentage", "Total_team", "Percentage_non_male", "Share_women",
        "Share_sexualized", "Avg_Reviews", "Release_year"]
corr = games[cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            linewidths=1, square=True, ax=ax)
ax.set_title("Spearman correlation between game-level variables (64 games)")
plt.tight_layout()
plt.show()

🔍 **Findings**<br>

- Strongest pair: Percentage_non_male and Share_women (0.69). These two measure nearly the same thing, so this one is expected and not a finding.

- Team_percentage and Release_year (0.36): more recent games have more women in the team.

- Team_percentage with Percentage_non_male (0.35) and with Share_women (0.30): more women in the team goes with more female characters. This matches the team-group result from the team-group comparison above.

- Share_sexualized is negative with Percentage_non_male (−0.26), Release_year (−0.24) and Share_women (−0.22): games with more female characters, and more recent games, have a lower share of sexualized characters. These are weak and about as strong as the noise level at n=64.

- Avg_Reviews is about 0 with everything: review scores don't track gender mix or sexualization here.

- ⚠️ **Caution:** 
- Correlation is not causation.
- The games are top-selling or top-rated ones, so there is a selection effect.
- Share_sexualized has many ties at 0.

# 📝 Conclusiones

> We answer our three questions with what the data shows. All numbers compare **women and men only** (596 characters); Non-binary (33) and Custom (8) characters are too few to compare. The games are top-selling or top-rated, so the results describe **successful games**, not the whole industry.

---

## Q1. Are women more sexualized than men, in what ways, and does it depend on the role?

**Answer: women are sexualized far more often than men, on every criterion, and this holds whatever their role.**



- **How much:** 
29 of 203 women (14.3%) have a sexualization score above 0, against 4 of 393 men (1.0%). That is about 14 times as often. Overall, only 34 of the 637 characters are sexualized, and nobody reaches the maximum score.

- **In what ways:** 
women are above men on all four criteria. The most frequent are sexualized clothing (17 women, 8.4%, against 2 men, 0.5%) and the damsel in distress role (15 women, 7.4%, against 1 man, 0.3%). Then come sexualized cutscenes (10 against 1) and the trophy role (6 against 1).

- **Does it depend on the role:** 
no. The gap appears in every role with enough characters: among protagonists (2 of 16 women against 1 of 45 men), main characters (11 of 78 against 1 of 120) and secondary characters (12 of 84 against 1 of 173). Grouped in three role tiers, the gap is the same in each: lead 2 of 16 women against 1 of 45 men, major 15 of 97 against 2 of 168, minor 12 of 90 against 1 of 180. So it is linked to gender more than to the role. No main antagonist or sidekick is sexualized, among women or men.

- **Over time:** 
the share of sexualized women fell from 24 of 117 (20.5%) in 2012–2017 to 5 of 86 (5.8%) in 2018–2022. This is a lower level in recent years, not a proven trend: each year has very few cases.

- ⚠️ **Caution:** 
the counts are small (34 sexualized characters in total, and a single man in most cells), so every percentage should be read together with its count.

---

## Q2. What roles do women hold: lead, playable, romantic interest?

**Answer: women are a smaller share of the protagonists, almost as often playable as men, and much more often a romantic interest.**

- **Lead roles:** 
women are 34% of the characters, but only 26% of the protagonists (16 of 61). They are better represented among main characters (39%, 78 of 198). Deuteragonists (8 of 18) and sidekicks (6 of 13) are too few to draw conclusions. Grouped in three tiers, women are 26% of the lead tier (protagonists, 16 of 61), 37% of the major tier (protagonists excluded: deuteragonists, main characters and main antagonists, 97 of 265) and 33% of the minor tier (sidekicks and secondary characters, 90 of 270), against 34% overall.

- **Playable:** 
33 of 203 women (16.3%) are playable, against 72 of 393 men (18.3%). Being playable is not where women are missing.

- **Romantic interest:** 
31 women (15.3%) are a romantic interest, against 16 men (4.1%), about 3.7 times as often. For women, 14 are an unavoidable partner and 17 an option the player can choose.

- **Team mix:** 
Games made by a team with at least one woman have more female characters (35% on average, against 27% in games with no woman in the team; 28 and 36 games). With three groups, the average share of women among the characters is 27% when the team has no woman (36 games), 33% with 1-25% women (20 games) and 41% with more than 25% (8 games): it rises step by step, but the last group is small. This is an association, and it does not prove a cause.

- **Over time:** 
the share of female characters stays between about 25% and 37% in most years, with a peak of 45% in 2021 (5 games). There is no clear trend.

- ⚠️ **Caution:** 
"lead" means protagonist in the strict reading, and women are under-represented there. In the broad reading (the major tier), they reach about their overall share. The major tier includes main antagonists and a very large group of main characters (202), and the metadata does not say how the author separates a protagonist from a main character, so the broad reading is less precise.

---

## Q3. Good vs. bad: are women allies, antagonists or main villains?

**Answer: women are mostly allies of the protagonist, and rarely villains.**

- **Side:** 
153 women (75.4%) are on the protagonist's side, against 241 men (61.3%). Only 20 women (9.9%) are antagonists, against 109 men (27.7%). So women are 15.5% of the antagonists, while they are 34% of the characters.

- **Main villains:** 
11 of the 49 main antagonists (22.4%) are women, and none of these 11 is sexualized.

- **Both sides:** 
30 women (14.8%) and 43 men (10.9%) can be on either side. It is a small group.

- ⚠️ **Caution:** 
20 female antagonists is a small number. We did not look at time or team mix for this question.

---

## What it means

Across the three questions, the same picture appears: in these successful games, women are fewer than men in leading and villain roles.
They are far more often shown in a sexualized or romantic way, and they are mostly on the hero's side. 
Games made by teams that include women show more female characters. 
These are descriptions of a small set of 64 games, so they are a starting point, not a proof.

- **Angle B and `Team_percentage` is heavily skewed.** 36 of the 64 games (56%) have exactly 0% women in the team, and the median is 0%. Correlating or averaging the raw percentage is therefore misleading. Instead, compare two groups: games with **0%** vs games with **>0%** (n = 36 vs 28), and show `n` for both. If the >0% group is still informative, it can be split further (for example, 1–25% and >25%).